hsi experiment does the following. It first does a small hyperparameter search on the transformed validation set(manually only due to high overhead) only PC-KNN considered.
Unfortunately here i used Random Search using sobol even though WCD lattice makes more sense as we only search over a single dim.
I corrected this for the final eval together with found hyperparameters of PC-KNN.  We also comapre the final result that uses reduction to 3x3 followed by random projection to collapsing into a single image and found it to improve results.

We als create a trade of figure when gating is used.

In [ ]:
#TODO if time rerun this using wcd.

In [ ]:
import torch

from src.utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
dataset ="si_score_resnet"
architecture = "resnet50_pretrained"

In [ ]:

from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data)


In [ ]:
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
print(len(dataset_train))

In [ ]:
print(len(train_loader))


In [ ]:
dataset_train[2][0].shape

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]

In [ ]:

from model.get_model import get_network
from model.train import train_and_get_model
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")

model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

#if si score then no train
if not "si_score" in dataset:
    train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
            "accelerator": "auto",
            "max_epochs": 100,
            "precision": "16-mixed",
    },load_if_exists=True)


#res = evaluate_base_model(model, test_loader_transformed, device)
#print(res)
model.eval().cuda()

In [ ]:
list(model.named_modules())

In [ ]:
#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
import confidence.direct.logit_based
import src.utils.affine_transforms

energy = confidence.direct.logit_based.EnergyConfidence()
from src.utils.transform_sequence import TransformSequence

transformation_sequence = TransformSequence(
    transformations=[
        src.utils.affine_transforms.AffineTransformation2D.ROTATION.value,
    ],
    domains=[(-torch.pi, torch.pi)],
    device=device,init_method="sobol"
)


In [ ]:
test_loader_transformed = torch.utils.data.DataLoader(test_loader_transformed.dataset, batch_size=8, shuffle=False, num_workers=4,pin_memory=True,persistent_workers=True)

In [ ]:
transform_seq = transformation_sequence.cuda()

In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, 240, 240), device=device.type)

In [ ]:

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule

In [ ]:
model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_siscore")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
transform_name ="rotation"

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision
from matplotlib.backends.backend_pdf import PdfPages

def unnormalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    mean = torch.tensor(mean,device=img.device).view(3, 1, 1)
    std = torch.tensor(std,device=img.device).view(3, 1, 1)
    return img * std + mean


def show_and_save_batch(loader, title, nrow=4, pdf_prefix="output"):
    imgs, _ = next(iter(loader))

    # Unnormalize
    imgs_unnorm = unnormalize(imgs)

    # Ensure enough images for a good grid
    if len(imgs_unnorm) < nrow * nrow:
        imgs2, _ = next(iter(loader))
        imgs_unnorm = torch.cat([imgs_unnorm, unnormalize(imgs2)], dim=0)

    # Build image grid
    grid = torchvision.utils.make_grid(
        imgs_unnorm[:nrow * nrow], nrow=nrow
    ).permute(1, 2, 0).cpu().clip(0, 1)

    # Show
    plt.figure(figsize=(10, 10))
    plt.imshow(grid)
    plt.axis("off")
    plt.show()

    # Save grid as ONE PDF, using your savepath()
    pdf_file = savepath(f"{pdf_prefix}_{title}.pdf")
    #remove json ending
    pdf_file = pdf_file.replace(".json", "")
    with PdfPages(pdf_file) as pdf:
        fig = plt.figure(figsize=(10, 10))
        plt.imshow(grid)
        plt.axis("off")
        os.makedirs(os.path.dirname(pdf_file), exist_ok=True)
        pdf.savefig(fig)
        plt.close(fig)

    print(f"Saved PDF grid → {pdf_file}")


# ---- Use ----
show_and_save_batch(train_loader, "train", nrow=4, pdf_prefix="train")
show_and_save_batch(val_loader, "val", nrow=4, pdf_prefix="val")
show_and_save_batch(test_loader_transformed, "test", nrow=4, pdf_prefix="test")


In [ ]:
import torch
import torch.nn as nn
import numpy as np

class TestTimeAugmenter(nn.Module):
    """
    Wraps a model to perform Test-Time Augmentation (TTA).
    Uses a systematic linspace for cyclical rotations.
    """
    def __init__(self, model, transformation_problem, n_samples=16):
        super().__init__()
        self.model = model
        self.transformation_problem = transformation_problem
        self.n_samples = n_samples

    def forward(self, x):
        if self.training:
            return self.model(x)

        batch_size = x.shape[0]
        device = x.device

        combined_logits = self.model(x)


        angles = np.linspace(-np.pi, np.pi, self.n_samples, endpoint=False)
        if self.n_samples %2 ==0:
            zero_idx = np.abs(angles).argmin()
            assert -1e-8<np.abs(angles).min()<1e-8
            augmented_angles = np.delete(angles, zero_idx)

        for angle in augmented_angles:
            params = torch.full((batch_size, 1), angle, device=device)
            x_aug = self.transformation_problem.transform(x, params)
            combined_logits += self.model(x_aug)

        return combined_logits / self.n_samples


model_test_time = TestTimeAugmenter(model, transformation_sequence, n_samples=16)

import json, os

# 1. Transformed Model
if not os.path.exists(savepath("result_tta_transformed")):
    res = evaluate_base_model(model_test_time, test_loader_transformed, device)
    json.dump(res, open(savepath("result_tta_transformed"), 'w'))
print(json.load(open(savepath("result_tta_transformed"))))

# 2. Raw Model
if not os.path.exists(savepath("result_tta")):
    res = evaluate_base_model(model_test_time, test_loader, device)
    json.dump(res, open(savepath("result_tta"), 'w'))
print(json.load(open(savepath("result_tta"))))

In [ ]:

from search.random_search import RSLR

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)

In [ ]:
energy_confidence = SinglePassConfidence(model, EnergyConfidence())
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

res_energy = load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=TransformationProblem(energy_confidence, transform_seq,
                                                                  consolidate_method="consolidate_simple"),
    test_loader=val_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("normal_energy_val"),show_progress=True,
    repeats=1)

In [ ]:
firt_image = next(iter(train_loader_no_shuffle))[0].to(device)
plt.imshow(firt_image[1].permute(1, 2, 0).cpu())

In [ ]:
from get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)

In [ ]:
embedding_cache_path

In [ ]:
from model.get_model import get_network_layer
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 0)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)



from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule


nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)
print("Evaluating first layer")
print("Evaluating first layer")
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed,
    max_batch_override=32,
    save_path=savepath("layer0_perclassknn_val"),show_progress=True,
    repeats=1)

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)
print(embeddings_t.shape)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule



nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)
print("Evaluating second layer")
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed,
    max_batch_override=32,
    save_path=savepath("layer1_perclassknn_val"),show_progress=True,
    repeats=1)

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 2)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search


nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)
print("Evaluating second layer")
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed,
    max_batch_override=32,
    save_path=savepath("layer2_perclassknn_val"),show_progress=True,
    repeats=1)

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search



nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None,k=10)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()
from search.random_search import RSLR

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)
print("Evaluating second layer")
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed,
    max_batch_override=32,
    save_path=savepath("layer1_perclassknn_10_val"),show_progress=True,
    repeats=1)

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search



nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()
print(embeddings_t.shape)
print(embeddings_t.shape)

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)
print("Evaluating second layer")
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed,
    max_batch_override=32,
    save_path=savepath("layer1_perclassknn_val_mixed"),show_progress=True,
    repeats=1)

In [ ]:
list(model.named_modules())

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True,reducer_select="collapse_image")
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True,reducer_select="collapse_image")




nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
print(embeddings_t.shape)
print(embeddings_t.shape)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

di = RSLR(
    initial_samples=17,
    local_runs=1,
    local_max_steps=0,
)
print("Evaluating second layer")
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=val_loader_transformed,
    max_batch_override=32,
    save_path=savepath("layer1_perclassknn_val_mixed_image_collapse"),show_progress=True,
    repeats=1)

In [ ]:
#mixed first layer was best for our single run so we use it here

Evaluation with found hyperparameters. I also swtiched to WCD as we only have a single dimension so random search from before
did not make that much sense as even spacing should help cover all regions. I did not rerun the above with WCD for time reason.

In [ ]:
from search.coordinate_descent import WeightedCoordinateDescent
from search.objective_generators import WCD_LATTICE_WRAPPER

wcd = WeightedCoordinateDescent(
    rounds=1,samples_per_dim=[17,])

di = WCD_LATTICE_WRAPPER(wcd)

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name

layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search



nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed,
    max_batch_override=32,
    save_path=savepath("final_result"),show_progress=True,
    repeats=4)


In [ ]:
energy_confidence = SinglePassConfidence(model, EnergyConfidence())
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

res_energy = load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=TransformationProblem(energy_confidence, transform_seq,
                                                                consolidate_method="consolidate_simple"),
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("energy_final"),show_progress=True,
    repeats=4)
print(res_energy)


The following compares image collapse and a reduction to 3x3 followed by random projection. It was not used for selecting the hyperparams.

In [ ]:
cache_train = train_cache
embeddings_t, final_t, classes_t = cache_train.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True, concat=True,reducer_select="collapse_image")
dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=True, flatten=True,reducer_select="collapse_image")

from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule


nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed_image_collapse"),show_progress=True,
    repeats=1)




load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed_image_collapse"),show_progress=True,
    repeats=2)

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("final_results_mixed_image_collapse"),show_progress=True,
    repeats=4)

In [ ]:
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name



layer,layer_io = get_network_layer(dataset_info, architecture, 1)

embeddings_t, final_t, classes_t = train_cache.get_correct_embeddings(layer, capture_modes=layer_io, flatten=True)
dual_output_model = train_cache.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)

print(dual_output_model.feature_reducers)
from src.utils.eval.ood_performance import evaluate_confidence_module, evaluate_confidence_and_search

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence,PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule


nn_pytorch_pretrained = PerClassKNNConfidence(metric="mixed", input_transform=None,mixed_alpha=0.5)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.000)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple",max_batch_size=32)
model.cuda().eval()

load_or_run_evaluate_confidence_and_search(
    model, optimizer=di, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed,
    max_batch_override=32,
    save_path=savepath("final_result"),show_progress=True,
    repeats=4)


The results are that reducing to a single pixel removes information we can use for canonicalization

In [ ]:
from typing import Optional
def visualize_canonicalization_grid(
    model,
    optimizer,
    problem,
    test_loader,
    n_rows: int = 4,
    n_cols: int = 4,
    device: str = "cuda",
    save_prefix: Optional[str] = None
):
    """
    Visualize images before and after canonicalization in a 4x4 grid.
    Green border = correct classification, Red border = incorrect.

    Args:
        save_prefix: If provided, saves {prefix}_before.pdf and {prefix}_after.pdf
    """
    model.eval()
    n_samples = n_rows * n_cols

    # Get a batch of data
    data, target = next(iter(test_loader))
    data, target = data.to(device), target.to(device)
    if len(data) < n_samples:
        data2, target2 = next(iter(test_loader))
        data = torch.cat([data, data2.to(device)], dim=0)
        target = torch.cat([target, target2.to(device)], dim=0)


    data = data[:n_samples]
    target = target[:n_samples]

    # Optimize transformations (canonicalize)
    res = optimizer.optimize(problem, data, y=target)
    params = res[0]
    x_canonicalized = problem.transform(data, params)

    # Get predictions
    with torch.no_grad():
        pred_before = model(data).argmax(dim=-1)
        pred_after = model(x_canonicalized).argmax(dim=-1)

    # unnormalize for visualization
    data = unnormalize(data)
    x_canonicalized = unnormalize(x_canonicalized)
    def add_border(img_tensor, is_correct):
        """Add colored border to image (C, H, W)"""
        color = torch.tensor([0.0, 0.45, 0.70] if is_correct else [0.85, 0.35, 0.25], device=img_tensor.device)
        bordered = img_tensor.clone()

        # Handle grayscale
        if bordered.shape[0] == 1:
            bordered = bordered.repeat(3, 1, 1)

        # Add border (2 pixel width)
        border_width = 2
        for c in range(3):
            bordered[c, :border_width, :] = color[c]
            bordered[c, -border_width:, :] = color[c]
            bordered[c, :, :border_width] = color[c]
            bordered[c, :, -border_width:] = color[c]

        return bordered

    # Create bordered images
    data_bordered = torch.stack([add_border(data[i], pred_before[i] == target[i]) for i in range(n_samples)])
    canon_bordered = torch.stack([add_border(x_canonicalized[i], pred_after[i] == target[i]) for i in range(n_samples)])

    # Create grids
    grid_before = torchvision.utils.make_grid(data_bordered.cpu(), nrow=n_cols, padding=2)
    grid_after = torchvision.utils.make_grid(canon_bordered.cpu(), nrow=n_cols, padding=2)

    # Display side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(grid_before.permute(1, 2, 0))
    axes[0].set_title('Before Canonicalization', fontsize=14)
    axes[0].axis('off')
    axes[1].imshow(grid_after.permute(1, 2, 0))
    axes[1].set_title('After Canonicalization', fontsize=14)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Save separate PDFs if requested
    if save_prefix:
        # Before
        fig_before, ax_before = plt.subplots(1, 1, figsize=(8, 8))
        ax_before.imshow(grid_before.permute(1, 2, 0))
        ax_before.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_before.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_before)

        # After
        fig_after, ax_after = plt.subplots(1, 1, figsize=(8, 8))
        ax_after.imshow(grid_after.permute(1, 2, 0))
        ax_after.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_after.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_after)

        print(f"Saved: {save_prefix}_before.pdf and {save_prefix}_after.pdf")

    # Print statistics
    acc_before = (pred_before == target).float().mean().item()
    acc_after = (pred_after == target).float().mean().item()

    print(f"Accuracy - Before: {acc_before:.3f}, After: {acc_after:.3f}")

    return {"accuracy_before": acc_before, "accuracy_after": acc_after}

visualize_canonicalization_grid(
    model=model,
    optimizer=di,
    problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=savepath("canonicalization_grid").replace(".json", "")
)

In [ ]:
from src.utils.eval.vis import plt_setup_paper

# --- Tradeoff utilities (copy into `experiment_thesis/imagenet_resnet.ipynb`) ---
import os
import numpy as np
import torch
import tqdm
import matplotlib.pyplot as plt

# Ensure result dirs
results_dir_trade = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_tradeoff")
os.makedirs(results_dir_trade, exist_ok=True)
os.makedirs(os.path.dirname(os.path.join(embedding_cache_path, dataset, architecture, transform_name, "dummy")), exist_ok=True)

def default_cache_path(label: str):
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(embedding_cache_path, dataset, architecture, transform_name, f"{safe}_cached_confidences.npz")

@torch.no_grad()
def calculate_confidences_and_cache(
    optimizer,
    best_problem,
    conf_fn,
    test_loader,
    test_loader_transformed,
    device,
    n_runs: int = 5,
    cache_path: str = "cached_confidences.npz",
    force_recompute: bool = False,
    augment_true_func=None,
):
    if os.path.exists(cache_path) and not force_recompute:
        print(f" Loading cached results from {cache_path}")
        return dict(np.load(cache_path, allow_pickle=True))

    print("=== Calculating base confidences (no optimization) ===")

    def eval_conf(loader, augment_func=None):
        all_conf, correct = [], []
        for x_test, y_test in tqdm.tqdm(loader):
            x_test = x_test.to(device)
            if augment_func is not None:
                x_test = augment_func(x_test)
            conf_values, logits = conf_fn(x_test)
            if logits is None:
                logits = model(x_test)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct.append((pred_y == y_test).numpy())
            all_conf.append(conf_values.cpu().numpy())
        return np.concatenate(correct), np.concatenate(all_conf)

    correct_predicted, all_conf = eval_conf(test_loader, augment_true_func)
    correct_predicted_transformed, all_conf_transformed = eval_conf(test_loader_transformed)

    results = {
        "correct_predicted": correct_predicted,
        "all_conf": all_conf,
        "correct_predicted_transformed": correct_predicted_transformed,
        "all_conf_transformed": all_conf_transformed,
    }

    print("\n=== Running probabilistic optimizer ===")

    optimized_correct_runs, optimized_conf_runs = [], []
    optimized_correct_runs_t, optimized_conf_runs_t = [], []

    for run_idx in range(n_runs):
        print(f"\n--- Optimization run {run_idx + 1}/{n_runs} ---")

        correct_run, conf_run = [], []
        for x_test, y_test in tqdm.tqdm(test_loader, desc=f"Test run {run_idx+1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_optimized = best_problem.transform_sequence.transform(x_test, p)
            conf_values, logits = conf_fn(x_optimized)
            if logits is None:
                logits = model(x_optimized)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run.append((pred_y == y_test).numpy())
            conf_run.append(conf_values.cpu().numpy())
        optimized_correct_runs.append(np.concatenate(correct_run))
        optimized_conf_runs.append(np.concatenate(conf_run))

        correct_run_t, conf_run_t = [], []
        for x_test, y_test in tqdm.tqdm(test_loader_transformed, desc=f"Transformed run {run_idx+1}"):
            x_test = x_test.to(device)
            p, error, _ = optimizer.optimize(best_problem, x_test)
            x_optimized = best_problem.transform_sequence.transform(x_test, p)
            conf_values, logits = conf_fn(x_optimized)
            if logits is None:
                logits = model(x_optimized)
            pred_y = torch.argmax(logits, dim=1).cpu()
            correct_run_t.append((pred_y == y_test).numpy())
            conf_run_t.append(conf_values.cpu().numpy())
        optimized_correct_runs_t.append(np.concatenate(correct_run_t))
        optimized_conf_runs_t.append(np.concatenate(conf_run_t))

    results["optimized_correct_predicted"] = np.stack(optimized_correct_runs, axis=1)
    results["optimized_all_conf"] = np.stack(optimized_conf_runs, axis=1)
    results["optimized_correct_predicted_transformed"] = np.stack(optimized_correct_runs_t, axis=1)
    results["optimized_all_conf_transformed"] = np.stack(optimized_conf_runs_t, axis=1)

    np.savez_compressed(cache_path, **results)
    print(f"\nCached results saved to: {cache_path}")
    return results

W =plt_setup_paper()
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def plot_confidence_threshold_results(
    results: dict,
    dataset_name: str = "Dataset",
    use_only_improved: bool = True,
    improvement_metric: str = "confidence",
    relative_improvement: float = 0.0,
    ci_multiplier: float = 1.96,
    lower_quantile: float = 0.001,
    upper_quantile: float = 1.0,
    plot_ratio=True,
    save_path=None,
    ratio_name="Optimization Ratio",
    threshold_transform=lambda x: x,
):
    if improvement_metric != "confidence":
        raise ValueError("Only improvement_metric='confidence' is supported.")

    raw_conf_t = results["all_conf_transformed"]
    raw_conf_o = results["all_conf"]

    # Calculate quantiles in original space
    ql_raw = np.quantile(raw_conf_t, lower_quantile)
    qh_raw = np.quantile(raw_conf_t, upper_quantile)

    # Check if the transform flips the order (monotonically decreasing)
    # We test with the quantiles themselves
    ql_trans = threshold_transform(ql_raw)
    qh_trans = threshold_transform(qh_raw)
    is_decreasing = ql_trans > qh_trans

    all_conf = threshold_transform(results["all_conf"])
    all_conf_t = threshold_transform(results["all_conf_transformed"])
    opt_conf = threshold_transform(results["optimized_all_conf"])
    opt_conf_t = threshold_transform(results["optimized_all_conf_transformed"])

    corr = results["correct_predicted"]
    corr_t = results["correct_predicted_transformed"]
    opt_corr = results["optimized_correct_predicted"]
    opt_corr_t = results["optimized_correct_predicted_transformed"]

    # Transform range for relative improvement scaling
    min_conf_trans = float(min(all_conf.min(), all_conf_t.min()))
    max_conf_trans = float(max(all_conf.max(), all_conf_t.max()))
    conf_range_trans = abs(max_conf_trans - min_conf_trans)

    # We iterate through the ORIGINAL scale to ensure we pick the same
    # samples as the original graph, then map the threshold for the x-axis
    raw_threshold_steps = np.linspace(ql_raw, qh_raw, 400)
    plot_thresholds = threshold_transform(raw_threshold_steps)

    n_runs = opt_conf.shape[1]
    acc_orig_runs, acc_trans_runs = [], []
    ratio_orig_runs, ratio_trans_runs = [], []

    for r in range(n_runs):
        acc_o, acc_t = [], []
        ratio_o, ratio_t = [], []

        for i in range(len(raw_threshold_steps)):
            raw_thr = raw_threshold_steps[i]
            trans_thr = plot_thresholds[i]

            # CRITICAL: If transform is decreasing, 'conf <= raw_thr'
            # is equivalent to 'trans_conf >= trans_thr'
            if is_decreasing:
                applied_o = all_conf >= trans_thr
                applied_t = all_conf_t >= trans_thr
            else:
                applied_o = all_conf <= trans_thr
                applied_t = all_conf_t <= trans_thr

            # Comparison Logic (using transformed values)
            # We assume 'higher' in transformed space means 'better' for these checks
            # If the transform flips confidence, you might need to flip these too
            improved_o = opt_conf[:, r] >= all_conf if not is_decreasing else opt_conf[:, r] <= all_conf
            improved_t = opt_conf_t[:, r] >= all_conf_t if not is_decreasing else opt_conf_t[:, r] <= all_conf_t

            if relative_improvement > 0.0:
                rel_imp_o = abs(opt_conf[:, r] - all_conf) / conf_range_trans
                rel_imp_t = abs(opt_conf_t[:, r] - all_conf_t) / conf_range_trans
                improved_o = np.logical_and(improved_o, rel_imp_o >= relative_improvement)
                improved_t = np.logical_and(improved_t, rel_imp_t >= relative_improvement)

            attempted_o = np.logical_and(applied_o, improved_o)
            attempted_t = np.logical_and(applied_t, improved_t)

            ratio_o.append(applied_o.mean())
            ratio_t.append(applied_t.mean())

            use_opt_o = attempted_o if use_only_improved else applied_o
            use_opt_t = attempted_t if use_only_improved else applied_t

            final_corr_o = np.where(use_opt_o, opt_corr[:, r], corr)
            final_corr_t = np.where(use_opt_t, opt_corr_t[:, r], corr_t)

            acc_o.append(final_corr_o.mean())
            acc_t.append(final_corr_t.mean())

        acc_orig_runs.append(acc_o)
        acc_trans_runs.append(acc_t)
        ratio_orig_runs.append(ratio_o)
        ratio_trans_runs.append(ratio_t)

    # Convert to numpy for CI calculation
    acc_orig_runs, acc_trans_runs = np.array(acc_orig_runs), np.array(acc_trans_runs)
    ratio_orig_runs, ratio_trans_runs = np.array(ratio_orig_runs), np.array(ratio_trans_runs)

    def compute_ci(y):
        n = y.shape[0]
        mean = y.mean(axis=0)
        se = y.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.zeros_like(mean)
        return mean, mean - ci_multiplier * se, mean + ci_multiplier * se

    fig, ax1 = plt.subplots(figsize=(W, W/3.1))
    tab10 = plt.get_cmap("tab10")
    col_o, col_t, col_ro, col_rt = tab10(0), tab10(1), tab10(2), tab10(3)

    ax1.grid(True, linestyle="--", alpha=0.6)

    m_o, lo_o, hi_o = compute_ci(acc_orig_runs)
    m_t, lo_t, hi_t = compute_ci(acc_trans_runs)

    ax1.plot(plot_thresholds, m_o, label="Acc. Original", color=col_o)
    ax1.fill_between(plot_thresholds, lo_o, hi_o, alpha=0.25, color=col_o)
    ax1.plot(plot_thresholds, m_t, label="Acc. Transformed", color=col_t)
    ax1.fill_between(plot_thresholds, lo_t, hi_t, alpha=0.25, color=col_t)

    ax1.set_xlabel("Threshold (Recalculated)", fontsize=9)
    ax1.set_ylabel("Accuracy", fontsize=9)

    if plot_ratio:
        ax2 = ax1.twinx()
        rm_o, rlo_o, rhi_o = compute_ci(ratio_orig_runs)
        rm_t, rlo_t, rhi_t = compute_ci(ratio_trans_runs)
        ax2.plot(plot_thresholds, rm_o, linestyle="--", color=col_ro, label="Ratio Original")
        ax2.plot(plot_thresholds, rm_t, linestyle="--", color=col_rt, label="Ratio Transformed")
        ax2.set_ylabel(ratio_name)


    ax1.set_xlim([ql_trans, qh_trans])
    if plot_ratio:
        ax2.set_xlim([ql_trans, qh_trans])

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path,bbox_inches="tight")
    plt.show()

    print(f"[{dataset_name}] Direction: {'Decreasing' if is_decreasing else 'Increasing'}")
    print(f"Range: {ql_trans:.4f} to {qh_trans:.4f}")


In [ ]:
from torchvision import models
from dataset.si_score import ImageNetSubset

# Use ViT preprocessing for SI-Score dataset
vit_transform = models.ResNet50_Weights.IMAGENET1K_V2.transforms()




imagenet_dataset_val = ImageNetSubset(root="D:/imagenet_full_val",
                                  transform=vit_transform)

loader_in = torch.utils.data.DataLoader(imagenet_dataset_val, batch_size=32, shuffle=False, num_workers=20,
                                              pin_memory=True, persistent_workers=True,prefetch_factor=4)


In [ ]:
wcd = WeightedCoordinateDescent(
    rounds=1,samples_per_dim=[17,])

di = WCD_LATTICE_WRAPPER(wcd)

In [ ]:




problem_to_use = problem_nn_pytorch_pretrained



conf_fn = problem_to_use.confidence_module







cache_file = default_cache_path("imagenet_tradeoff2")
results = calculate_confidences_and_cache(
    optimizer=di,
    best_problem=problem_to_use,
    conf_fn=conf_fn,
    test_loader=loader_in,
    test_loader_transformed=test_loader_transformed,
    device=device,
    n_runs=1,
    cache_path=cache_file,
)

plot_confidence_threshold_results(results, dataset_name=dataset, save_path=os.path.join(results_dir_trade, "confidence_threshold.pdf"),use_only_improved=True,relative_improvement=0.1,threshold_transform=lambda y:1/y-1)
print("Saved plot to:", os.path.join(results_dir_trade, "confidence_threshold.pdf"))

In [ ]:
from typing import Optional
def visualize_canonicalization_grid(
    model,
    optimizer,
    problem,
    test_loader,
    n_rows: int = 4,
    n_cols: int = 4,
    device: str = "cuda",
    save_prefix: Optional[str] = None
):
    """
    Visualize images before and after canonicalization in a 4x4 grid.
    Green border = correct classification, Red border = incorrect.

    Args:
        save_prefix: If provided, saves {prefix}_before.pdf and {prefix}_after.pdf
    """
    model.eval()
    n_samples = n_rows * n_cols

    # Get a batch of data
    data, target = next(iter(test_loader))
    data, target = data.to(device), target.to(device)
    if len(data) < n_samples:
        data2, target2 = next(iter(test_loader))
        data = torch.cat([data, data2.to(device)], dim=0)
        target = torch.cat([target, target2.to(device)], dim=0)


    data = data[:n_samples]
    target = target[:n_samples]

    # Optimize transformations (canonicalize)
    res = optimizer.optimize(problem, data, y=target)
    params = res[0]
    x_canonicalized = problem.transform(data, params)

    # Get predictions
    with torch.no_grad():
        pred_before = model(data).argmax(dim=-1)
        pred_after = model(x_canonicalized).argmax(dim=-1)

    # unnormalize for visualization
    data = unnormalize(data)
    x_canonicalized = unnormalize(x_canonicalized)
    def add_border(img_tensor, is_correct):
        """Add colored border to image (C, H, W)"""
        color = torch.tensor([0, 1, 0] if is_correct else [1, 0, 0], device=img_tensor.device)
        bordered = img_tensor.clone()

        # Handle grayscale
        if bordered.shape[0] == 1:
            bordered = bordered.repeat(3, 1, 1)

        # Add border (2 pixel width)
        border_width = 2
        for c in range(3):
            bordered[c, :border_width, :] = color[c]
            bordered[c, -border_width:, :] = color[c]
            bordered[c, :, :border_width] = color[c]
            bordered[c, :, -border_width:] = color[c]

        return bordered

    # Create bordered images
    data_bordered = torch.stack([add_border(data[i], pred_before[i] == target[i]) for i in range(n_samples)])
    canon_bordered = torch.stack([add_border(x_canonicalized[i], pred_after[i] == target[i]) for i in range(n_samples)])

    # Create grids
    grid_before = torchvision.utils.make_grid(data_bordered.cpu(), nrow=n_cols, padding=2)
    grid_after = torchvision.utils.make_grid(canon_bordered.cpu(), nrow=n_cols, padding=2)

    # Display side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(grid_before.permute(1, 2, 0))
    axes[0].set_title('Before Canonicalization', fontsize=14)
    axes[0].axis('off')
    axes[1].imshow(grid_after.permute(1, 2, 0))
    axes[1].set_title('After Canonicalization', fontsize=14)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Save separate PDFs if requested
    if save_prefix:
        # Before
        fig_before, ax_before = plt.subplots(1, 1, figsize=(8, 8))
        ax_before.imshow(grid_before.permute(1, 2, 0))
        ax_before.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_before.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_before)

        # After
        fig_after, ax_after = plt.subplots(1, 1, figsize=(8, 8))
        ax_after.imshow(grid_after.permute(1, 2, 0))
        ax_after.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(f"{save_prefix}_after.pdf", bbox_inches='tight', pad_inches=0)
        plt.close(fig_after)

        print(f"Saved: {save_prefix}_before.pdf and {save_prefix}_after.pdf")

    # Print statistics
    acc_before = (pred_before == target).float().mean().item()
    acc_after = (pred_after == target).float().mean().item()

    print(f"Accuracy - Before: {acc_before:.3f}, After: {acc_after:.3f}")

    return {"accuracy_before": acc_before, "accuracy_after": acc_after}

visualize_canonicalization_grid(
    model=model,
    optimizer=di,
    problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed,
    n_rows=4,
    n_cols=4,
    device=device,
    save_prefix=savepath("canonicalization_grid").replace(".json", "")
)